# 16 — 数式変更Capstone

上流コードを触る前に、変更したい仮定・式・テスト・比較指標を一つの変更票にします。

**前提**: `15_tuning_laboratory.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebook_pympc":
    ROOT = ROOT.parent
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
if str(PYMPC_ROOT) not in sys.path:
    sys.path.insert(0, str(PYMPC_ROOT))

os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
os.environ.setdefault("MUJOCO_GL", "egl")
print("workspace :", ROOT)
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 例: 異方性摩擦

現行はx/yで同じ \(\mu\) を使います。異方性地面を研究するなら候補式は

\[
|F_x|\le\mu_xF_z,\qquad |F_y|\le\mu_yF_z
\]

です。変更箇所は制約式だけでなく、設定値、runtime parameter、可視化、
feasibility test、baseline比較まで及びます。

In [2]:
import numpy as np
def isotropic(force, mu):
    fx, fy, fz = force
    return abs(fx) <= mu*fz and abs(fy) <= mu*fz and fz >= 0
def anisotropic(force, mu_x, mu_y):
    fx, fy, fz = force
    return abs(fx) <= mu_x*fz and abs(fy) <= mu_y*fz and fz >= 0

cases = [
    np.array([20., 10., 50.]),
    np.array([10., 20., 50.]),
    np.array([20., 20., 50.]),
]
for f in cases:
    print(f, "old:", isotropic(f, .42), "new:", anisotropic(f, .6, .25))

[20. 10. 50.] old: True new: True
[10. 20. 50.] old: True new: False
[20. 20. 50.] old: True new: False


## 変更票

1. **仮説**: どの現象を既存式が表現できないか
2. **数式差分**: 旧式と新式、単位、domain
3. **コード差分**: model / OCP / config / interfaceの境界
4. **単体テスト**: 既知のfeasible/infeasible点
5. **退行テスト**: \(\mu_x=\mu_y\) なら旧式と一致
6. **閉ループ比較**: 同seed、同指令、同scene
7. **棄却条件**: solver失敗、飽和、計算時間悪化の上限

このNotebookでは候補式だけを試し、上流コードは変更していません。

In [3]:
rng = np.random.default_rng(3)
forces = rng.uniform([-30,-30,0], [30,30,80], size=(10000,3))
old = np.array([isotropic(f,.42) for f in forces])
same = np.array([anisotropic(f,.42,.42) for f in forces])
print("regression mismatches:", np.count_nonzero(old != same))
assert np.array_equal(old, same)

regression mismatches: 0


## 修了判定

`simulation.py` の観測から `compute_actions`、WB参照、MPC、GRF、
stance/swing torque、MuJoCo stepまでをshape・単位・frame付きで説明でき、
1パラメータ変更をログで評価し、上の変更票を埋められれば修了です。

次の研究候補は、input-rate MPC、sampling MPC、foothold optimization、
nonuniform discretization、integral states、RTIの順に、標準nominalとの差分として学びます。

# Repository Performance Benchmark — 30シナリオ実測

## 目的と方法

READMEの機能一覧ではなく、**現行リポジトリのnominal acados NMPC、WBInterface、stance/swing torque、90% torque clip、MuJoCo Go2 plantを実際に閉ループ実行**し、どこまで安定・追従できるかを測ります。

- Easy 10条件: 平地、静止〜0.30 m/s、後退・横移動・yawを1要因ずつ
- Normal 10条件: 0.40–0.60 m/s、旋回、低摩擦、random boxes、Perlin
- Hard 10条件: 0.80–1.00 m/s、速度+旋回、強い横移動、凸凹・坂・低摩擦
- Easy/Normalは5秒、Hardは6秒
- `type='nominal'`, `N=12`, `dt_mpc=0.02 s`, 100 Hz update
- 各条件を別processで起動し、controller stateや設定値の持越しを防止

結果の正本は`benchmark_results/scenario_results.json`、再実行コードは`../scripts/run_pympc_curriculum_benchmark.py`です。

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Notebookをどこから開いても結果JSONを見つける。
HERE = Path.cwd().resolve()
if HERE.name != "notebook_pympc":
    HERE = HERE / "notebook_pympc"
RESULTS = HERE / "benchmark_results" / "scenario_results.json"
payload = json.loads(RESULTS.read_text(encoding="utf-8"))

# scenario定義と実測metricsを横持ちにして30行へ変換。
rows = []
for entry in payload["results"]:
    row = {**entry["scenario"], **entry["metrics"]}
    rows.append(row)
df = pd.DataFrame(rows)
assert len(df) == 30
assert df.groupby("difficulty").size().to_dict() == {"easy": 10, "hard": 10, "normal": 10}
print("loaded:", len(df), "real closed-loop scenarios")

## 成功判定と計測量

`success`は次をすべて満たす場合です。

1. MuJoCoがterminationせず規定時間を完走
2. base heightが0.16 m以上
3. 最大|roll|と|pitch|が35 deg以下
4. 水平速度RMSEが`max(0.18, 0.55×指令速度)`以下
5. yaw-rate RMSEが`max(0.20, 0.75×|指令yaw|)`以下

性能値として速度RMSE、姿勢最大値、torque飽和率、control時間、acados solve時間、実時間倍率、solver statusを保存しています。成功閾値は論文の保証値ではなく、この教材で比較可能にするため事前定義した判定です。

In [ ]:
# 30条件の一覧。ID、地形、指令、成功、主要性能を同じ行で確認する。
columns = [
    "id", "difficulty", "description", "scene", "vx", "vy", "yaw_rate",
    "success", "stable", "speed_rmse_mps", "max_abs_roll_deg",
    "max_abs_pitch_deg", "torque_saturation_rate", "mpc_solve_p95_ms",
]
display(df[columns].round(3))

In [ ]:
# 難易度別に成功率、追従、計算時間、飽和を集約する。
order = ["easy", "normal", "hard"]
summary = df.groupby("difficulty").agg(
    scenarios=("id", "count"),
    successes=("success", "sum"),
    stable_runs=("stable", "sum"),
    mean_speed_rmse_mps=("speed_rmse_mps", "mean"),
    mean_torque_saturation=("torque_saturation_rate", "mean"),
    mean_mpc_p95_ms=("mpc_solve_p95_ms", "mean"),
    mean_realtime_factor=("realtime_factor", "mean"),
).reindex(order)
summary["success_rate"] = summary["successes"] / summary["scenarios"]
display(summary.round(3))

In [ ]:
# 成功/失敗と性能劣化をID単位で可視化する。
colors = df["success"].map({True: "#16a34a", False: "#dc2626"})
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
axes[0].bar(df["id"], df["speed_rmse_mps"], color=colors)
axes[0].set_ylabel("speed RMSE [m/s]")
axes[0].set_title("30 real PyMPC scenarios (green=success, red=failure)")
axes[1].bar(df["id"], df["torque_saturation_rate"]*100, color=colors)
axes[1].set_ylabel("torque saturation [%]")
axes[2].bar(df["id"], df["mpc_solve_p95_ms"], color=colors)
axes[2].axhline(10.0, color="k", ls="--", label="100 Hz budget = 10 ms")
axes[2].set_ylabel("MPC solve p95 [ms]")
axes[2].set_xlabel("scenario ID")
axes[2].legend()
for ax in axes:
    ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

## 実測から得られた結論

- **Easy: 10/10成功**。平均速度RMSEは約0.034 m/s、torque飽和は0%。小さい平地指令では安定して追従しました。
- **Normal: 8/10成功、9/10は姿勢・高さを維持**。random boxesのN05は転倒しないものの追従閾値を外れ、N06は0.35 m/sで姿勢限界を超えました。平均RMSEは約0.087 m/sです。
- **Hard: 7/10成功**。平地1.0 m/s (H02)、random boxes 0.5 m/s (H06)、Perlin低摩擦0.45 m/s (H07)が失敗しました。平均RMSE約0.299 m/s、平均torque飽和率約1.44%まで悪化しました。
- acados solve p95の難易度別平均は約1.97 / 2.14 / 2.48 msで、全条件で100 Hzの10 ms予算内でした。今回の失敗は平均計算時間超過より、速度・地形・摩擦に対する姿勢悪化とtorque飽和の増加に対応します。
- 全MPC updateのsolver statusは2でした。標準設定がSQP反復上限1なので「最大反復到達」を毎回返します。上流実装はstatus 1/4だけをfallback対象にし、status 2のiterateを使用します。したがって、短いsolve時間を「数値的完全収束」と解釈してはいけません。

**性能に関する結論:** 現行baselineは低〜中速平地と保守的なPerlin/坂条件に強い一方、高速平地とrandom boxes・低摩擦の複合条件で余裕が急減します。次のチューニング対象は、失敗条件のgait/footholdとtorque飽和、そしてSQP反復数と閉ループ性能の交換です。

## 限界と再実行

この結果はGo2 MuJoCo、seed固定、5–6秒、nominal controllerの結果です。実機性能、長時間安定性、統計的成功確率、sampling MPCとの優劣を直接示しません。custom bumpy地形は教材側のterrain登録を使います。また実行環境にGNU makeが無いため、配布済みacados shared objectを再利用しました。OCP・solver呼出し・制御ループは上流実装そのものですが、数式を変更した場合はsolverを再buildできる環境が必要です。

30条件を再実行するコマンド:

```bash
source .env.workshop
.venv/bin/python scripts/run_pympc_curriculum_benchmark.py
```

比較を研究結果として扱う場合は、各条件を複数seed・30秒以上へ拡張し、平均だけでなく信頼区間とfailure timeを報告してください。

## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。